In [ ]:
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
plt.rcParams['figure.facecolor'] = 'white'
import matplotlib.colors as colors
import cmocean.cm as cmo
from glob import glob
%config InlineBackend.print_figure_kwargs = {'bbox_inches': None}

In [ ]:
def prepro(ds):
    return ds.isel(y=slice(800, None))

Load grid and data files

In [ ]:
grid_files = ["/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mask.nc", 
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_hgr.nc",
              "/data0/project/drakkar/CONFIGS/CREG12.L75/GRID/CREG12.L75-REF08_mesh_zgr.nc"]

In [ ]:
grid = xr.open_mfdataset(grid_files, parallel=True, preprocess=prepro)

In [ ]:
u_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-UV_clim/" 
                                + "CREG12.L75-REF08_*.5d_U2Dclim.nc"))
u_data_filesFUT = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-FUT08-UV_clim/" 
                                + "CREG12.L75-FUT08_*.5d_U2Dclim.nc"))
v_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-UV_clim/" 
                                + "CREG12.L75-REF08_*.5d_V2Dclim.nc"))
v_data_filesFUT = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-FUT08-UV_clim/" 
                                + "CREG12.L75-FUT08_*.5d_V2Dclim.nc"))
ice_data_filesREF = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-REF08-icemod_clim/" 
                                + "CREG12.L75-REF08_*.5d_icemodclim.nc"))
ice_data_filesFUT = sorted(glob("/data0/project/drakkar/USERS/jrieck/CREG12.L75-FUT08-icemod_clim/" 
                                + "CREG12.L75-FUT08_*.5d_icemodclim.nc"))

In [ ]:
uREF = xr.open_mfdataset(u_data_filesREF, parallel=True, preprocess=prepro)
uFUT = xr.open_mfdataset(u_data_filesFUT, parallel=True, preprocess=prepro)
vREF = xr.open_mfdataset(v_data_filesREF, parallel=True, preprocess=prepro)
vFUT = xr.open_mfdataset(v_data_filesFUT, parallel=True, preprocess=prepro)
iceREF = xr.open_mfdataset(ice_data_filesREF, parallel=True, preprocess=prepro)
iceFUT = xr.open_mfdataset(ice_data_filesFUT, parallel=True, preprocess=prepro)

Compute wind speed and stresses and interpolate to T-grid

In [ ]:
utau_iceREF = uREF.utau_iceoce * iceREF.siconc
vtau_iceREF = vREF.vtau_iceoce * iceREF.siconc
utau_iceFUT = uFUT.utau_iceoce * iceFUT.siconc
vtau_iceFUT = vFUT.vtau_iceoce * iceFUT.siconc

In [ ]:
utau_atmREF = uREF.utau_atmoce * (1 - iceREF.siconc)
vtau_atmREF = vREF.vtau_atmoce * (1 - iceREF.siconc)
utau_atmFUT = uFUT.utau_atmoce * (1 - iceFUT.siconc)
vtau_atmFUT = vFUT.vtau_atmoce * (1 - iceFUT.siconc)

In [ ]:
utau_totalREF = utau_iceREF + utau_atmREF
vtau_totalREF = vtau_iceREF + vtau_atmREF
utau_totalFUT = utau_iceFUT + utau_atmFUT
vtau_totalFUT = vtau_iceFUT + vtau_atmFUT

In [ ]:
utau_totalT_REF = utau_totalREF.interp(x=np.arange(0.5, len(uREF.x), 1))
utau_totalT_FUT = utau_totalFUT.interp(x=np.arange(0.5, len(uFUT.x), 1))
vtau_totalT_REF = vtau_totalREF.interp(y=np.arange(0.5, len(vREF.y), 1))
vtau_totalT_FUT = vtau_totalFUT.interp(y=np.arange(0.5, len(vFUT.y), 1))

In [ ]:
tau_total_REF = ((utau_totalT_REF**2) + (vtau_totalT_REF**2))**0.5
tau_total_FUT = ((utau_totalT_FUT**2) + (vtau_totalT_FUT**2))**0.5

In [ ]:
uWINDT_REF = uWIND_REF.uwspd10.interp(x=np.arange(0.5, len(uREF.x), 1))
uWINDT_FUT = uWIND_FUT.vwspd10.interp(x=np.arange(0.5, len(uFUT.x), 1))
vWINDT_REF = vWIND_REF.uwspd10.interp(y=np.arange(0.5, len(vREF.y), 1))
vWINDT_FUT = vWIND_FUT.vwspd10.interp(y=np.arange(0.5, len(vFUT.y), 1))

In [ ]:
WIND_REF = ((uWINDT_REF**2) + (vWINDT_REF**2))**0.5
WIND_FUT = ((uWINDT_FUT**2) + (vWINDT_FUT**2))**0.5

Compute averages

In [ ]:
WIND_REF_plot = WIND_REF.mean("time_counter").squeeze().compute()
uWINDT_REF_plot = uWINDT_REF.mean("time_counter").squeeze().compute()
vWINDT_REF_plot = vWINDT_REF.mean("time_counter").squeeze().compute()
ice_REF_plot = iceREF.siconc.mean("time_counter").squeeze().compute()
tau_total_REF_plot = tau_total_REF.mean("time_counter").squeeze().compute()
utau_totalT_REF_plot = utau_totalT_REF.mean("time_counter").squeeze().compute()
vtau_totalT_REF_plot = vtau_totalT_REF.mean("time_counter").squeeze().compute()

WIND_FUT_plot = WIND_FUT.mean("time_counter").squeeze().compute()
uWINDT_FUT_plot = uWINDT_FUT.mean("time_counter").squeeze().compute()
vWINDT_FUT_plot = vWINDT_FUT.mean("time_counter").squeeze().compute()
ice_FUT_plot = iceFUT.siconc.mean("time_counter").squeeze().compute()
tau_total_FUT_plot = tau_total_FUT.mean("time_counter").squeeze().compute()
utau_totalT_FUT_plot = utau_totalT_FUT.mean("time_counter").squeeze().compute()
vtau_totalT_FUT_plot = vtau_totalT_FUT.mean("time_counter").squeeze().compute()

Plot Fig. S6

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
qs = 50

ax[0, 0].set_facecolor("silver")
a = ax[0, 0].pcolormesh(flxREF.x, flxREF.y, WIND_REF_plot, 
                        cmap=cmo.amp, vmin=0, vmax=5, shading="nearest", zorder=1)
cb0 = plt.colorbar(a, ax=ax[0, 0])
cb0.set_label(r"m s$^{-1}$", labelpad=2)
ax[0, 0].contour(iceREF.x, iceREF.y, ice_REF_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q0 = ax[0, 0].quiver(flxREF.x[0::qs], flxREF.y[0::qs],
                     uWINDT_REF_plot[0::qs, 0::qs], vWINDT_REF_plot[0::qs, 0::qs],
                     scale=50)
q0k = ax[0, 0].quiverkey(q0, 0.87, 0.72, 2, r'2 m s$^{-1}$', labelpos='S',
                         coordinates='axes')
ax[0, 0].text(50, 900, "1996-2015", fontsize=14)
ax[0, 0].set_ylabel("y")

ax[0, 1].set_facecolor("silver")
b = ax[0, 1].pcolormesh(flxFUT.x, flxFUT.y, WIND_FUT_plot, 
                        cmap=cmo.amp, vmin=0, vmax=5, shading="nearest", zorder=1)
cb1 = plt.colorbar(b, ax=ax[0, 1])
cb1.set_label(r"m s$^{-1}$", labelpad=2)
ax[0, 1].contour(iceFUT.x, iceFUT.y, ice_FUT_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q1 = ax[0, 1].quiver(flxFUT.x[0::qs], flxFUT.y[0::qs],
                     uWINDT_FUT_plot[0::qs, 0::qs], vWINDT_FUT_plot[0::qs, 0::qs],
                     scale=50)
q1k = ax[0, 1].quiverkey(q1, 0.87, 0.72, 2, r'2 m s$^{-1}$', labelpos='S',
                         coordinates='axes')
ax[0, 1].text(50, 900, "2051-2070", fontsize=14)

ax[0, 2].set_facecolor("silver")
c = ax[0, 2].pcolormesh(flxFUT.x, flxFUT.y, WIND_FUT_plot - WIND_REF_plot, 
                        cmap=cmo.balance, vmin=-0.4, vmax=0.4, shading="nearest", zorder=1)
cb2 = plt.colorbar(c, ax=ax[0, 2])
cb2.set_label(r"m s$^{-1}$", labelpad=0)
ax[0, 2].contour(iceFUT.x, iceFUT.y, ice_FUT_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q2= ax[0, 2].quiver(flxFUT.x[0::qs], flxFUT.y[0::qs],
                    (uWINDT_FUT_plot[0::qs, 0::qs] - uWINDT_REF_plot[0::qs, 0::qs]),
                    (vWINDT_FUT_plot[0::qs, 0::qs] - vWINDT_REF_plot[0::qs, 0::qs]),
                    scale=20)
q2k = ax[0, 2].quiverkey(q2, 0.87, 0.72, 1, r'1 m s$^{-1}$', labelpos='S',
                         coordinates='axes')
ax[0, 2].text(50, 900, "FUT - REF", fontsize=14)

ax[1, 0].set_facecolor("silver")
a = ax[1, 0].pcolormesh(flxREF.x, flxREF.y, tau_total_REF_plot, 
                        cmap=cmo.amp, vmin=0, vmax=0.07, shading="nearest", zorder=1)
cb0 = plt.colorbar(a, ax=ax[1, 0])
cb0.set_label(r"N m$^{-2}$", labelpad=1)
ax[1, 0].contour(iceREF.x, iceREF.y, ice_REF_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q0 = ax[1, 0].quiver(flxREF.x[0::qs], flxREF.y[0::qs],
                     utau_totalT_REF_plot[0::qs, 0::qs], vtau_totalT_REF_plot[0::qs, 0::qs],
                     scale=1)
q0k = ax[1, 0].quiverkey(q0, 0.87, 0.72, 0.1, r'0.1 N m$^{-2}$', labelpos='S',
                      coordinates='axes')
ax[1, 0].set_ylabel("y")

ax[1, 1].set_facecolor("silver")
b = ax[1, 1].pcolormesh(flxFUT.x, flxFUT.y, tau_total_FUT_plot, 
                        cmap=cmo.amp, vmin=0, vmax=0.07, shading="nearest", zorder=1)
cb1 = plt.colorbar(b, ax=ax[1, 1])
cb1.set_label(r"N m$^{-2}$", labelpad=1)
ax[1, 1].contour(iceFUT.x, iceFUT.y, ice_FUT_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q1 = ax[1, 1].quiver(flxFUT.x[0::qs], flxFUT.y[0::qs],
                     utau_totalT_FUT_plot[0::qs, 0::qs], vtau_totalT_FUT_plot[0::qs, 0::qs],
                     scale=1)
q1k = ax[1, 1].quiverkey(q1, 0.87, 0.72, 0.1, r'0.1 N m$^{-2}$', labelpos='S',
                         coordinates='axes')

ax[1, 2].set_facecolor("silver")
c = ax[1, 2].pcolormesh(flxFUT.x, flxFUT.y, tau_total_FUT_plot - tau_total_REF_plot, 
                        cmap=cmo.balance, vmin=-0.01, vmax=0.01, shading="nearest", zorder=1)
cb2 = plt.colorbar(c, ax=ax[1, 2])
cb2.set_label(r"N m$^{-2}$", labelpad=0)
ax[1, 2].contour(iceFUT.x, iceFUT.y, ice_FUT_plot, 
                 levels=[0.15, 0.8], colors=["cornflowerblue", "royalblue"], zorder=2)
q2= ax[1, 2].quiver(flxFUT.x[0::qs], flxFUT.y[0::qs],
                    (utau_totalT_FUT_plot[0::qs, 0::qs] - utau_totalT_REF_plot[0::qs, 0::qs]),
                    (vtau_totalT_FUT_plot[0::qs, 0::qs] - vtau_totalT_REF_plot[0::qs, 0::qs]),
                    scale=0.3)
q2k = ax[1, 2].quiverkey(q2, 0.87, 0.72, 0.02, r'0.02 N m$^{-2}$', labelpos='S',
                         coordinates='axes')

for axx in [ax[0, 1], ax[0, 2], ax[1, 1], ax[1, 2]]:
    axx.set_ylabel("")
    axx.set_yticklabels([])

for axx in [ax[1, 0], ax[1, 1], ax[1, 2]]:
    axx.set_xlabel("x")
    
[xx.text(0, 1050, t, fontsize=14) for xx, t in zip([ax[0, 0], ax[0, 1], ax[0, 2], ax[1, 0], ax[1, 1], ax[1, 2]], 
                                                    ["a)", "b)", "c)", "d)", "e)", "f)"])]

fig.text(0.5, 0.95, r"10 m atmospheric wind", fontsize=18, ha="center");
fig.text(0.5, 0.49, r"surface stress", fontsize=18, ha="center");

plt.subplots_adjust(top=0.92, wspace=0.1, left=0.05, right=0.9, hspace=0.47)

plt.savefig("figures/Figure_S6_wind_stress.png", dpi=300)